# m6a_epitranscriptome — Pan-cancer alteration-frequency retrieval

Reproducible cBioPortal pipeline. Retrieves somatic mutation (MAF) and discrete copy-number (GISTIC) alteration frequencies for the m6a_epitranscriptome gene set across the 32 TCGA PanCancer Atlas studies, and writes all data tables to `../data/`. Run cells in order (`pip install -r ../requirements.txt`).

In [ ]:
# m6a_epitranscriptome — cBioPortal retrieval pipeline
SLUG  = "m6a_epitranscriptome"
GENES = ["METTL3", "METTL14", "METTL16", "WTAP", "VIRMA", "ZC3H13", "RBM15", "FTO", "ALKBH5", "YTHDF1", "YTHDF2", "YTHDF3", "YTHDC1", "YTHDC2", "IGF2BP1", "IGF2BP2", "IGF2BP3"]
import requests, pandas as pd, os
BASE="https://www.cbioportal.org/api"; H={"Accept":"application/json"}
def GET(u):
    r=requests.get(u,headers=H,timeout=90); r.raise_for_status(); return r.json()
def POST(u,body):
    r=requests.post(u,headers={**H,"Content-Type":"application/json"},json=body,timeout=240); r.raise_for_status(); return r.json()
print("cBioPortal:", GET(f"{BASE}/info").get("portalVersion"))

## Steps 1–2 · Studies and per-study sample/profile map

In [ ]:
# Steps 1-2: enumerate TCGA PanCancer Atlas studies; per study build the
# sequenced∩cna sample intersection (the analysis denominator N) and find the
# MAF mutation profile and discrete GISTIC CNA profile.
studies=[]; pn=0
while True:
    d=GET(f"{BASE}/studies?projection=SUMMARY&pageSize=1000&pageNumber={pn}&direction=ASC&sortBy=studyId")
    if not d: break
    studies+=d; pn+=1
    if len(d)<1000: break
pan=pd.DataFrame(studies)
pan=pan[pan['studyId'].str.contains('_tcga_pan_can_atlas_2018',case=False,na=False)].sort_values('studyId').reset_index(drop=True)
print(len(pan),"PanCancer Atlas studies")
infra={}
for _,r in pan.iterrows():
    sid=r['studyId']; sls=GET(f"{BASE}/studies/{sid}/sample-lists")
    def best(suf):
        b=set()
        for s in sls:
            if s['sampleListId'].endswith(suf):
                ids=set(GET(f"{BASE}/sample-lists/{s['sampleListId']}/sample-ids"))
                if len(ids)>len(b): b=ids
        return b
    seq=best('_sequenced'); cna=best('_cna'); inter=sorted(seq&cna)
    mps=GET(f"{BASE}/studies/{sid}/molecular-profiles")
    mut=[m['molecularProfileId'] for m in mps if m['molecularAlterationType']=='MUTATION_EXTENDED' and m['datatype']=='MAF']
    gis=[m['molecularProfileId'] for m in mps if m['molecularAlterationType']=='COPY_NUMBER_ALTERATION' and m['datatype']=='DISCRETE']
    infra[sid]={'acr':sid.split('_tcga')[0].upper(),'ctype':r['cancerTypeId'],'sname':r['name'],
                'inter':inter,'N':len(inter),'seqN':len(seq),'cnaN':len(cna),'mut':mut[0] if mut else None,'gis':gis[0] if gis else None}
print("Total N =", sum(d['N'] for d in infra.values()))

## Step 3 · Gene → Entrez mapping

In [ ]:
# Step 3: map HGNC symbols -> Entrez IDs
gm=POST(f"{BASE}/genes/fetch?geneIdType=HUGO_GENE_SYMBOL&projection=SUMMARY", GENES)
sym2ez={g['hugoGeneSymbol']:g['entrezGeneId'] for g in gm}; ez2sym={v:k for k,v in sym2ez.items()}
ez=[sym2ez[g] for g in GENES if g in sym2ez]
print("mapped", len(ez), "/", len(GENES), "genes")

## Step 4 · Retrieve mutations + CNA → master long table

In [ ]:
# Step 4: per study, fetch mutations (MAF) and discrete CNA (GISTIC) for the gene
# set restricted to the intersection samples; compute per-gene counts/frequencies.
# CNA codes: amp=+2, gain=+1, shallow_del=-1, deep_del=-2. any_alter = mutation ∪ any-CNA.
rows=[]
for sid,d in sorted(infra.items(), key=lambda kv:kv[1]['acr']):
    N=d['N']; S=set(d['inter'])
    muts=POST(f"{BASE}/molecular-profiles/{d['mut']}/mutations/fetch?projection=SUMMARY", {"entrezGeneIds":ez,"sampleIds":d['inter']})
    mg={}
    for m in muts:
        g=ez2sym.get(m['entrezGeneId'])
        if g and m['sampleId'] in S: mg.setdefault(g,set()).add(m['sampleId'])
    cnas=POST(f"{BASE}/molecular-profiles/{d['gis']}/molecular-data/fetch?projection=SUMMARY", {"entrezGeneIds":ez,"sampleIds":d['inter']})
    cg={g:{'amp':0,'gain':0,'sdel':0,'ddel':0,'any':set()} for g in GENES}
    for c in cnas:
        g=ez2sym.get(c['entrezGeneId'])
        if g is None or c['sampleId'] not in S: continue
        v=int(c['value'])
        if v==2: cg[g]['amp']+=1
        elif v==1: cg[g]['gain']+=1
        elif v==-1: cg[g]['sdel']+=1
        elif v==-2: cg[g]['ddel']+=1
        if v!=0: cg[g]['any'].add(c['sampleId'])
    for g in GENES:
        ms=mg.get(g,set()); cs=cg[g]['any']; aa=ms|cs
        rows.append(dict(tumour_acronym=d['acr'],studyId=sid,studyName=d['sname'],geneSymbol=g,entrezGeneId=sym2ez.get(g,''),
            N=N,mut_n=len(ms),mut_freq=len(ms)/N,amp_n=cg[g]['amp'],amp_freq=cg[g]['amp']/N,gain_n=cg[g]['gain'],gain_freq=cg[g]['gain']/N,
            shallow_del_n=cg[g]['sdel'],shallow_del_freq=cg[g]['sdel']/N,deep_del_n=cg[g]['ddel'],deep_del_freq=cg[g]['ddel']/N,
            any_cna_n=len(cs),any_cna_freq=len(cs)/N,any_alter_n=len(aa),any_alter_freq=len(aa)/N,
            mut_only_n=len(ms-cs),cna_only_n=len(cs-ms),mut_and_cna_n=len(ms&cs)))
master=pd.DataFrame(rows); os.makedirs("../data",exist_ok=True)
master.to_csv(f"../data/tcga_pancan_{SLUG}_alteration_freqs_long.csv", index=False)
print("master", master.shape)

## Step 5 · Derived tables

In [ ]:
# Step 5: derived tables (matrices, gene ranking, top-5, tumour burden, BRCA deep-dive, sample summary)
acrs=sorted(master['tumour_acronym'].unique())
for metric,fn in [('any_alter_freq','any_alter'),('any_cna_freq','any_cna'),('mut_freq','mut')]:
    master.pivot(index='geneSymbol',columns='tumour_acronym',values=metric).reindex(GENES)[acrs].reset_index()\
        .to_csv(f"../data/tcga_pancan_{SLUG}_{fn}_freq_matrix.csv", index=False)
gr=master.groupby('geneSymbol').agg(mean_any_alter_freq=('any_alter_freq','mean'),mean_mut_freq=('mut_freq','mean'),
    mean_any_cna_freq=('any_cna_freq','mean'),mean_amp_freq=('amp_freq','mean'),mean_deep_del_freq=('deep_del_freq','mean'),
    max_any_alter_freq=('any_alter_freq','max'),n_studies=('any_alter_freq','size')).reset_index().sort_values('mean_any_alter_freq',ascending=False)
gr.insert(0,'rank',range(1,len(gr)+1)); gr.to_csv(f"../data/{SLUG}_gene_rank_summary.csv", index=False)
t5=[]
for g in GENES:
    for i,(_,r) in enumerate(master[master.geneSymbol==g].nlargest(5,'any_alter_freq').iterrows(),1):
        t5.append(dict(geneSymbol=g,rank=i,tumour_acronym=r['tumour_acronym'],N=r['N'],any_alter_freq=r['any_alter_freq'],
                       mut_freq=r['mut_freq'],any_cna_freq=r['any_cna_freq'],amp_freq=r['amp_freq'],deep_del_freq=r['deep_del_freq']))
pd.DataFrame(t5).to_csv(f"../data/{SLUG}_top5_tumours_per_gene.csv", index=False)
bd=master.groupby(['tumour_acronym','studyId']).agg(mean_any_alter_freq=('any_alter_freq','mean'),mean_mut_freq=('mut_freq','mean'),
    mean_any_cna_freq=('any_cna_freq','mean'),N=('N','first'),n_genes=('geneSymbol','size')).reset_index().sort_values('mean_any_alter_freq',ascending=False)
bd.insert(0,'rank',range(1,len(bd)+1))
bd[['rank','tumour_acronym','mean_any_alter_freq','mean_mut_freq','mean_any_cna_freq','N','n_genes']].to_csv(f"../data/{SLUG}_tumour_burden_ranking.csv", index=False)
master[master.tumour_acronym=='BRCA'][['geneSymbol','entrezGeneId','mut_n','amp_n','gain_n','shallow_del_n','deep_del_n','any_cna_n','any_alter_n','N','mut_freq','amp_freq','deep_del_freq','any_cna_freq','any_alter_freq']]\
    .rename(columns={'geneSymbol':'gene','entrezGeneId':'entrez_id'}).to_csv(f"../data/brca_{SLUG}_alteration_frequencies.csv", index=False)
ss=pd.DataFrame([dict(studyId=s,cancerTypeId=d['ctype'],studyName=d['sname'],sampleCount_sequenced=d['seqN'],sampleCount_cna=d['cnaN'],
    intersection_sequenced_cna_n=d['N'],ratio_intersection_over_min_seq_cna=round(d['N']/min(d['seqN'],d['cnaN']),6)) for s,d in infra.items()])
ss.to_csv("../data/tcga_pancan_atlas_sample_set_summary.csv", index=False)
with open(f"../data/tcga_low_coverage_tumours_for_{SLUG}.txt","w") as f:
    f.write("Tumour types with N<100 intersection samples (interpret with caution):\n")
    for _,r in ss[ss.intersection_sequenced_cna_n<100].sort_values('intersection_sequenced_cna_n').iterrows():
        f.write(f"  {r['studyId'].split('_tcga')[0].upper()}: N={r['intersection_sequenced_cna_n']}\n")
print("All derived tables written to ../data/")